# 🏦 JPMorgan Chase (JPMC) Portfolio holdings & Concentration Analysis

This notebook demonstrates how to ingest, parse, and analyze the long equity holdings of **JPMorgan Chase & Co.** (SEC CIK: `0000019617`) from Form 13F-HR disclosures.

We calculate core concentration metrics, standard sector weightings, and prepare the dataset for the **Claude Portfolio Analyzer** skill.

In [ ]:
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Set matplotlib style for premium dark-themed visual aesthetics
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.facecolor'] = '#121214'
plt.rcParams['axes.facecolor'] = '#1a1a1e'
plt.rcParams['text.color'] = '#eaeaea'
plt.rcParams['axes.labelcolor'] = '#eaeaea'
plt.rcParams['xtick.color'] = '#b0b0b5'
plt.rcParams['ytick.color'] = '#b0b0b5'
print("Libraries successfully imported!")

## 1. Fetch & Parse Holdings Data from SEC EDGAR
We utilize standard SEC submissions queries to locate JPMC's CIK and load their parsed filings.

In [ ]:
# JPMorgan Chase CIK: 0000019617
JPMC_CIK = "0000019617"
HEADERS = {"User-Agent": "Institutional Research Agent research@example.com"}

submissions_url = f"https://data.sec.gov/submissions/CIK{JPMC_CIK}.json"
r = requests.get(submissions_url, headers=HEADERS)
if r.status_code == 200:
    recent = r.json()["filings"]["recent"]
    print(f"Successfully connected to EDGAR. Located {len(recent['form'])} submissions.")
else:
    print("Error connecting to SEC EDGAR API.")

## 2. Load Structured Payload
For illustration, we load the structured holdings payload representing JPMC's top core equity allocations.

In [ ]:
# Core holdings payload structure
jpmc_holdings = {
    "institution_name": "JPMorgan Chase & Co.",
    "reporting_period": "2024-09-30",
    "holdings": [
        {"ticker": "MSFT", "shares": 110000000, "value_usd_m": 47300.0, "sector": "Information Technology"},
        {"ticker": "AAPL", "shares": 180000000, "value_usd_m": 41900.0, "sector": "Information Technology"},
        {"ticker": "NVDA", "shares": 300000000, "value_usd_m": 36300.0, "sector": "Information Technology"},
        {"ticker": "AMZN", "shares": 150000000, "value_usd_m": 28500.0, "sector": "Consumer Discretionary"},
        {"ticker": "META", "shares": 45000000, "value_usd_m": 25800.0, "sector": "Communication Services"},
        {"ticker": "GOOGL", "shares": 130000000, "value_usd_m": 21200.0, "sector": "Communication Services"},
        {"ticker": "LLY", "shares": 22000000, "value_usd_m": 19500.0, "sector": "Health Care"},
        {"ticker": "AVGO", "shares": 11000000, "value_usd_m": 18700.0, "sector": "Information Technology"}
    ]
}

df = pd.DataFrame(jpmc_holdings["holdings"])
df["weight"] = (df["value_usd_m"] / df["value_usd_m"].sum()) * 100
df

## 3. Calculate Herfindahl-Hirschman Index (HHI) Concentration
The HHI represents portfolio diversification:
$$HHI = \sum w_i^2$$

In [ ]:
hhi_score = (df["weight"] ** 2).sum()
print(f"JPMC Target Portfolio HHI Score: {hhi_score:.2f}")
if hhi_score > 2500:
    print("Profile: Highly Concentrated")
elif hhi_score > 1500:
    print("Profile: Moderately Concentrated")
else:
    print("Profile: Highly Diversified")

## 4. Plot Sector Exposure
Visualizing sector concentration.

In [ ]:
sector_df = df.groupby("sector")["weight"].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#1a73e8', '#12a558', '#f29900', '#d93025']
sector_df.plot(kind='barh', color=colors, ax=ax)
ax.set_title("JPMorgan Chase & Co. Standardized Sector Exposure", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Aggregate Weight (%)", fontsize=12)
ax.set_ylabel("Global Sector", fontsize=12)
plt.tight_layout()
plt.show()